# C/C++ 事前学習用 問題集（宿題）

半導体デザインハッカソンでは、FPGAを搭載した評価ボード（KV260）を使って物体検出を行います。このFPGAにはXilinx（現AMD）が開発した深層学習用専用ユニットであるDPU（Deep Learning Processing Unit）が構成され、CNN(Convolutional Neural Network)を高速に動作させることができます。このDPUを使って深層学習用ネットワークの１つであるYOLOv3をC++プログラムを使って駆動・コントロールします。具体的には「画像の読み込み → 前処理 → DPU推論（yolov3ライブラリ＝既存）→ 後処理 → 表示」という一連の処理を行います。

ハッカソン当日に配布する2つのサンプルを **読んで・改造できる** ようになるのが目標です。

- [`yolov3_video_series_prof.cpp`](https://github.com/takgto/cpp-lab/blob/main/kv260/yolov3_video_series_prof.cpp) … 各処理がどれくらい時間がかかるかを測る **プロファイリング版**（1フレームずつ直列で実行）。
- [`yolov3_video_study.cpp`](https://github.com/takgto/cpp-lab/blob/main/kv260/yolov3_video_study.cpp) … 上の結果をもとに **どう並列処理すれば速くなるか** を考えるための版。

各章の「解説」で引用しているコードは、この2ファイルからの抜粋です。問題を解くのに全文を読む必要はありませんが、興味があればリンク先を眺めてみてください（KV260 の DPU を動かすためのライブラリを使っているので、この Colab ではコンパイルできません）。

そのために必要なC++の基礎を、以下の8章で身につけてください。

- **対象レベル**：C/C++を少し触ったことがある人。知らない文法は **検索しながら** でOKです。
- **使う機材**：ブラウザだけ。この Colab 上で C++ をコンパイル・実行します。環境構築は要りません。
- **進め方**：各章は **解説 → 設問 → 設問の解答** の順に並んでいます。**設問を自分で解いてから**、解答を開いて答え合わせをしてください。

**目次**（各章は 解説 → 設問 → 設問の解答 の順）

- [第1章　コンパイルと実行、コマンドライン引数](#scrollTo=cpp_homework_05)
- [第2章　配列のように使える `std::vector`](#scrollTo=cpp_homework_11)
- [第3章　`std::pair`・`make_pair`・`auto`・型の別名](#scrollTo=cpp_homework_23)
- [第4章　値渡し・参照渡し・ポインタと `const`（性能の鍵）](#scrollTo=cpp_homework_30)
- [第5章　`std::chrono` で処理時間を測る（プロファイリングの核心）](#scrollTo=cpp_homework_46)
- [第6章　関数とグローバル変数・スコープ](#scrollTo=cpp_homework_50)
- [第7章　クラス・テンプレート・ファンクタ](#scrollTo=cpp_homework_60)
- [第8章　動的メモリと `unique_ptr`](#scrollTo=cpp_homework_70)

## はじめに：Colab で C++ を動かす

**上のセルから順に ▶ を押していく**だけです。

- コードセルの1行目にある `%%writefile test.cpp` は「このセルの中身を `test.cpp` というファイルとして保存する」という Colab の命令です。
- その次のセルの `!g++ ...` が、保存したファイルをコンパイルして実行します（`!` は「ターミナルのコマンドとして実行する」印）。

```bash
!g++ -std=c++17 test.cpp -o test && ./test
```

- `-std=c++17` … C++17の文法を使う指定。
- `&&` … 左のコマンドが成功したら右を実行（コンパイルが通ったときだけ実行する）。

**設問のセルは自由に書き換えて構いません。** 書き換えたら、そのセルと次の実行セルをもう一度 ▶ してください。
まずは下の2つのセルを実行して、C++ が動くことを確かめましょう。

In [ ]:
%%writefile hello.cpp
#include <iostream>
int main() {
    std::cout << "Hello, C++ on Colab!\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 hello.cpp -o hello && ./hello

# 第1章　コンパイルと実行、コマンドライン引数

## 解説

サンプルは `[実行プログラム] <引数1> <引数2>` の形で **2つの引数** を受け取って動きます。
引数は `main` の `argc` / `argv` で受け取ります。

- `argc` … 引数の個数。**プログラム名を含めて** 数えるので、引数2個なら `argc` は 3。
- `argv` … 文字列（`char*`）の配列。`argv[0]` はプログラム名そのもの、`argv[1]` 以降がユーザーの渡した引数。
- 数値として使いたいときは `std::stoi(argv[1])` などで変換します。

---

## 設問

次の `test1.cpp` を用意します。サンプルにならって、**引数が2個（モデルと動画）でなければ終了する** チェックを入れてあります。

```cpp
#include <iostream>
int main(int argc, char** argv) {
    if (argc != 3) {                       // プログラム名+引数2個=3 でなければ
        std::cout << "Usage: " << argv[0] << " <model> <video>\n";
        return -1;                         // 引数が足りないので終了
    }
    std::cout << "argc = " << argc << "\n";
    for (int i = 0; i < argc; i++)
        std::cout << "argv[" << i << "] = " << argv[i] << "\n";
    return 0;
}
```

### 問1

この `test1.cpp` をコンパイルして `test1` という実行ファイルを作り、実行するコマンドは？

### 問2

`./test1 hello 123` と実行したとき、`argc` はいくつ？ `argv[0]`, `argv[1]`, `argv[2]` はそれぞれ何になる？ また、引数を付けずに `./test1` だけで実行すると何が起きる？

### 問3

このプログラムが `if (argc != 3)` で引数の個数を確認してから処理を始めているのはなぜ？

**ヒント**：`g++ -std=c++17 test1.cpp -o test1`。`argv[0]` はプログラム名そのもの。

## 設問の解答

### 問1

`g++ -std=c++17 test1.cpp -o test1` でコンパイルして `test1` を作り、`./test1 hello 123` で実行します（Colab では先頭に `!` を付けます）。

```bash
g++ -std=c++17 test1.cpp -o test1     # コンパイル
./test1 hello 123                      # 実行
```

### 問2

`argc = 3`。`argv[0] = ./test1`（プログラム名）、`argv[1] = hello`、`argv[2] = 123`。引数は **プログラム名を含めて** 数えるので、ユーザーが渡した引数が2個なら `argc` は3になります。一方、引数を付けずに `./test1` だけで実行すると `argc` は1なので、`if (argc != 3)` に引っかかり、`Usage: ./test1 <model> <video>` と表示して `return -1;` で終了します（引数が足りないので処理に進みません）。

### 問3

引数が足りないまま `argv[2]` などにアクセスすると、存在しないものを読んでしまい異常終了やバグの原因になります。`if (argc != 3)` で弾いて `return -1;` するのは、**必要な入力（モデルと動画）が揃っているかを最初に確認する** ためです。本番のYOLOサンプルも同じ理由でモデルと動画の2引数をチェックしています。

次のコードを実行して確認できます

In [ ]:
%%writefile test1.cpp
#include <iostream>
int main(int argc, char** argv) {
    if (argc != 3) {                       // プログラム名+引数2個=3 でなければ
        std::cout << "Usage: " << argv[0] << " <model> <video>\n";
        return -1;                         // 引数が足りないので終了
    }
    std::cout << "argc = " << argc << "\n";
    for (int i = 0; i < argc; i++)
        std::cout << "argv[" << i << "] = " << argv[i] << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 test1.cpp -o test1 && ./test1 hello 123

In [ ]:
!./test1

- `argv` は文字列（`char*`）の配列です。数値として使いたいときは `std::stoi(argv[1])` などで変換します。
- 試しに引数を **3個** 付けて実行してみてください（`!./test1 a b c`）。これも `argc != 3` で弾かれます。

# 第2章　配列のように使える `std::vector`

## 解説

サンプルでは検出結果や出力データを `std::vector` で扱います（`vector<float> result(sizeOut);`、`vector<int8_t*> results = {result0, result1, result2};`、`vector<vector<float>> boxes;` など）。

`std::vector` は **要素数を後から増やせる配列** です。

- `std::vector` は `<vector>` ヘッダで定義されているので、先頭に `#include <vector>` を書きます。追加は `push_back`、個数は `.size()`、範囲for文は `for (auto x : v)`。
- `vector` は要素を **連続したメモリ領域** に並べます。要素が増えて入れ物が足りなくなると、より大きな領域を確保して **全部コピーして引っ越す**（再確保）という動きをします。

---

## 設問

### 問4

`int` を入れる空の `vector` を作り、`10, 20, 30` の3つを追加し、要素数を表示するには？

### 問5

その全要素を **範囲for文（range-based for）** で1つずつ表示するには？

### 問6

サンプルの `vector<float> result(sizeOut);` のように「最初から要素数 `n` 個（初期値0）で作る」書き方は？

### 問7

「**最初から要素数 `n` 個で確保する**（または `reserve(n)` する）」のと、「**`push_back` で1つずつ足していく**」のとでは、**速度** と **メモリ消費** の面でどう違う？ サンプルが出力サイズ分を最初に `vector<float> result(sizeOut);` で確保しているのはなぜ？

問4〜問6 は下のセルに書いて動かしてください。

In [ ]:
%%writefile ex2.cpp
#include <iostream>
#include <vector>
int main() {
    // 1. int を入れる空の vector を作り、10, 20, 30 を追加して要素数を表示する

    // 2. 全要素を範囲for文で1つずつ表示する

    // 3. 要素数 5 個（初期値 0）の vector<float> を作り、要素数を表示する

    return 0;
}

In [ ]:
!g++ -std=c++17 ex2.cpp -o ex2 && ./ex2

## 設問の解答

### 問4

`std::vector<int> v;` で空の vector を作り、`v.push_back(10);` のように追加します。要素数は `v.size()` です。

### 問5

`for (auto x : v) std::cout << x << " ";` のように範囲for文で回します。

### 問6

`std::vector<float> result(5);` のようにコンストラクタに個数を渡すと、5個の要素が初期値 0 で作られます。この 0 初期化は C++ の規格で保証されていて、コンパイラやバージョンによりません。一方、`float buf[5];`（ローカル配列）、`new float[5]`（第8章）、`malloc` で確保した領域は **初期化されない** ので、自分で 0 を入れる必要があります。`v.reserve(5)` も容量を確保するだけで要素は 0 個です。

問4〜問6 をまとめたプログラムが次のセルです。

In [ ]:
%%writefile ans2.cpp
#include <iostream>
#include <vector>
int main() {
    std::vector<int> v;          // 1. 空のvector
    v.push_back(10);
    v.push_back(20);
    v.push_back(30);
    std::cout << "size = " << v.size() << "\n";   // 3

    for (auto x : v)             // 2. 範囲for
        std::cout << x << " ";
    std::cout << "\n";

    std::vector<float> result(5);  // 3. 要素5個（初期値0.0）で作る
    std::cout << "result size = " << result.size() << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ans2.cpp -o ans2 && ./ans2

- `for (auto x : v)` は要素のコピーを受け取ります。大きな要素を書き換えたい/コピーを避けたいときは `for (auto& x : v)`（参照）を使います（第4章につながる考え方）。

### 問7

**最初から確保する vs `push_back` で1つずつ（速度・メモリ）**

`vector` は連続したメモリ領域に要素を並べます。`push_back` で増やしていくと、確保済みの領域（容量＝capacity）が一杯になるたびに、**より大きな領域を新たに確保し、既存の全要素をそこへコピーして引っ越し、古い領域を解放** します（再確保）。容量はだいたい1.5〜2倍ずつ増えるので、`n` 個入れる間に再確保が何回か起こり、そのたびにコピーが発生します。

- **速度**：要素数が分かっているなら、`vector<float> result(n);`（または空のまま `v.reserve(n);`）で **最初に一度だけ確保** しておくと、途中の再確保・コピーが起きないぶん速くなります。`push_back` を無計画に繰り返すと、再確保のたびのコピーで遅くなります。
- **メモリ**：`push_back` の再確保では「古い領域＋新しい領域」が一瞬同時に存在し、また容量は要素数ぴったりではなく **多めに確保される**（余りが出る）ため、ムダが出やすいです。最初から `n` 個で作れば、必要な分だけを一度に確保できます。
- **使い分け**：個数が事前に分かる（サンプルの「出力テンソルのサイズ」など）→ **最初に確保**。個数が動的に決まる・予測しにくい → `push_back`（できれば `reserve` で見込み量を予約）。サンプルが `vector<float> result(sizeOut);` としているのは、出力サイズが分かっているので再確保を避け、最初に過不足なく確保するためです。

次のセルで、`push_back` のたびに **容量（`capacity()`）がどう増えるか** を実際に観察できます。表示された行の数だけ「再確保＋全コピー」が起きています。

In [ ]:
%%writefile ans2b.cpp
#include <iostream>
#include <vector>
// push_back で増やしていくと、容量(capacity)が足りなくなるたびに「再確保＋全要素コピー」が起きる。
// capacity() の変化を観察する。
int main() {
    std::vector<int> v;
    std::size_t last = 0;
    for (int i = 0; i < 1000; i++) {
        v.push_back(i);
        if (v.capacity() != last) {             // 容量が変わった＝再確保が起きた
            last = v.capacity();
            std::cout << "size=" << v.size() << "  capacity=" << v.capacity() << "\n";
        }
    }

    std::vector<int> w(1000);                   // 最初から 1000 個で確保
    std::cout << "\nw: size=" << w.size() << "  capacity=" << w.capacity()
              << "  (再確保なし)\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ans2b.cpp -o ans2b && ./ans2b

**補足：`vector(n)` は 0 だが、`new[]` は 0 とは限らない**

`vector<float>(5)` と `new float[5]` の中身を並べて表示します。`new[]` 側は 0 が並ぶこともありますが、それはたまたまで、規格上は不定です（実行のたび・環境によって変わります）。サンプルの `new int8_t[size]` で確保したバッファも同じで、書き込む前に読んではいけません。

In [ ]:
%%writefile ans2c.cpp
#include <iostream>
#include <vector>
int main() {
    std::vector<float> v(5);          // 要素5個。規格で「0 で初期化される」と決まっている
    float* p = new float[5];          // 要素5個。初期化されない（何が入っているかは不定）

    std::cout << "vector<float> v(5) : ";
    for (int i = 0; i < 5; i++) std::cout << v[i] << " ";
    std::cout << "\nnew float[5]       : ";
    for (int i = 0; i < 5; i++) std::cout << p[i] << " ";   // 0 に見えることもあるが、保証はない
    std::cout << "\n";

    delete[] p;
    return 0;
}

In [ ]:
!g++ -std=c++17 ans2c.cpp -o ans2c && ./ans2c

# 第3章　`std::pair`・`make_pair`・`auto`・型の別名

## 解説

サンプルは「フレーム番号」と「画像」をひとまとめにして運ぶため、`typedef pair<int, Mat> imagePair;` と別名を付け、`auto pair = make_pair(idxInputImage++, img);` のように使います。

- `std::pair<A, B>` は **2つの値の組**。要素は `p.first` / `p.second`。
- `std::pair` は `<utility>` というヘッダで定義されているので、使うファイルの先頭に `#include <utility>` を書きます（`std::string` を使うなら `#include <string>` も必要です）。
- `std::make_pair(a, b)` で型を書かずに作れる。
- `auto` は「右辺から型を推論して」変数を宣言する書き方。型名が長いときに便利。
- `using 別名 = 型;`（または `typedef 型 別名;`）で **型に別名** を付けられる。

---

## 設問

### 問8

`std::pair<int, std::string>` を1つ作り（例：`1` と `"cat"`）、`.first` と `.second` を表示するには？

### 問9

変数の型を明示せず `auto` で受け取るように書き換えると？

### 問10

`using imagePair = std::pair<int, std::string>;` のように **型に別名を付けて** から、その別名で変数を宣言するには？（`typedef` でも同じことができます）

In [ ]:
%%writefile ex3.cpp
#include <iostream>
#include <utility>
#include <string>

// 3. ここで型に別名を付ける（using imagePair = ...;）

int main() {
    // 1. std::pair<int, std::string> を作り（1 と "cat"）、.first と .second を表示する

    // 2. auto と std::make_pair で受け取る

    // 3. 別名 imagePair で変数を宣言して使う

    return 0;
}

In [ ]:
!g++ -std=c++17 ex3.cpp -o ex3 && ./ex3

## 設問の解答

### 問8

`std::pair<int, std::string> p(1, "cat");` と作り、`p.first`（1）と `p.second`（"cat"）で取り出します。

### 問9

`auto q = std::make_pair(2, std::string("dog"));` と書けば、型を書かずに `std::pair<int, std::string>` として受け取れます。

### 問10

先頭で `using imagePair = std::pair<int, std::string>;` と別名を付け、`imagePair r = std::make_pair(3, std::string("bird"));` のように別名で宣言します。

問8〜問10 をまとめたプログラムが次のセルです。

In [ ]:
%%writefile ans3.cpp
#include <iostream>
#include <utility>
#include <string>

using imagePair = std::pair<int, std::string>;  // 3. 型に別名を付ける

int main() {
    std::pair<int, std::string> p(1, "cat");     // 1.
    std::cout << p.first << " " << p.second << "\n";

    auto q = std::make_pair(2, std::string("dog")); // 2. auto で受ける
    std::cout << q.first << " " << q.second << "\n";

    imagePair r = std::make_pair(3, std::string("bird")); // 別名を使う
    std::cout << r.first << " " << r.second << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ans3.cpp -o ans3 && ./ans3

- サンプルの `typedef pair<int, Mat> imagePair;` も、ここでの `using imagePair = ...;` と同じ「型に別名を付ける」操作です（`using` は新しい書き方）。
- `auto pair = make_pair(idxInputImage++, img);` のように、**型を書くのが面倒・長い** ときに `auto` と `make_pair` が便利です。

# 第4章　値渡し・参照渡し・ポインタと `const`（性能の鍵）

## 解説

画像データ（`cv::Mat`）は大きいので、関数に **コピーで渡す** と毎回複製が起きて遅くなります。サンプルは `const Mat& frame`（参照渡し）や `vart::Runner* runner`（ポインタ）を使って複製を避けています。一方 `frame.clone()` は **わざと複製** する場面です。

参考：プロファイリング版の `post_process` は `auto img = frame.clone();` してから描画して返します。study版の `post_process` は `Mat& img` を受け取り、複製せず **直接** 描き込みます。

- 参照は型のうしろに `&`。コピーを防ぎつつ書き換えも防ぐのが `const 型&`。

**参照とポインタはどう違うのか。** どちらも「実体をコピーせず指す」点は同じですが、性格が違います。
話を簡単にするため、`int x = 42;` という普通の変数に対して、参照 `int& r = x;` とポインタ `int* p = &x;` を作った場合で比べます（`&x` は「`x` のアドレス」）。

| | 参照 `int& r = x;` | ポインタ `int* p = &x;` |
|---|---|---|
| 正体 | `x` の **別名**（`r` と書けば `x` のこと） | `x` の **アドレスを入れる変数** |
| 「何も指していない」状態 | **あり得ない**（必ず実体を指す） | **あり得る**（`int* p = nullptr;`） |
| 指す先を後から変える | **できない**（`r` はずっと `x`） | **できる**（`p = &y;` で `y` を指す） |
| 中身の触り方 | `r` をそのまま使う（`r + 1`、`r = 5`） | `*p` で中身になる（`*p + 1`、`*p = 5`） |
| 配列・アドレス計算 | できない | できる（`p + 1`、`p[i]`） |

`vector` や `Mat` のようにメンバを持つ型では、ポインタ経由のメンバ呼び出し `(*p).size()` を `p->size()` と略記できます（サンプルの `runner->execute_async(...)` などがこの形）。参照なら `r.size()` とそのまま書けます。

**使い分けの原則**：「必ず存在する1つのものを渡す」なら **参照**。読むだけなら `const T&`、書き換えて返すなら `T&`。これが C++ の基本形で、迷ったら参照です。
**ポインタを使うのは、参照ではできないことが必要なとき** に限ります。

1. 「無い」場合がある（`nullptr` を渡して「省略」を表したい）
2. 指す先を途中で付け替える
3. 配列を「先頭アドレス＋個数」で扱う（`new int8_t[size]` や C 流の API。第8章）
4. C 流のライブラリ API がポインタを要求している（サンプルの `vart::Runner*` がこれ）

サンプルを見ると、この原則どおりになっています。`const Mat& frame` は「必ず1枚ある画像を読むだけ」なので参照。`vart::Runner* runner` は、DPU ライブラリ（VART）の API が `Runner*` を要求するため、`main` で `unique_ptr` が持っている runner を `runner.get()` で **生ポインタとして借りて** 渡しています（所有は `unique_ptr` のまま。第8章）。`int8_t* result0 = new int8_t[size0]` は配列の先頭アドレスです。

---

## 設問

### 問11

`void f(BigData d)`（値渡し）と `void f(const BigData& d)`（参照渡し）は、大きなデータを渡すときどちらが速い？ なぜ？

### 問12

引数に付いている `const` は何を表す？（関数の中でその引数に何ができなくなる？）

### 問13

study版の `post_process` が `clone()` せず `Mat&` に直接描くのは、なぜ都合が良い？（「速度」と「描いた結果を呼び出し側に反映する」の2点で説明）

### 問14

サンプルは画像を `const Mat& frame`（参照）で、DPU の runner を `vart::Runner* runner`（ポインタ）で受け取っています。**なぜ片方は参照で、片方はポインタ** なのでしょう？ もし `frame` をポインタ `const Mat* frame` にしたら、何が変わる（何が面倒になる）？

この章は考える問題です。答えを書いてから、解答のコードを動かして確かめてください。

## 設問の解答

### 問11

**参照渡し（`const BigData&`）の方が速い** です。値渡しは関数を呼ぶたびにデータ全体を **複製** しますが、参照渡しは「実体を指す」だけなので複製が起きません。画像のような大きなデータでは差が大きくなります。

### 問12

`const` は「関数の中で **その引数を書き換えない**」という約束です。誤って変更してしまうミスを防ぎ、読み手にも「これは入力専用」と伝わります。

### 問13

study版の `post_process(Mat& img, ...)` が `clone()` しない理由は2つ：(1) **複製のコストを省ける**（速い）、(2) `Mat&`（非constの参照）で受け取り **元の画像に直接** 検出枠を描くので、**描いた結果がそのまま呼び出し側のフレームに反映** されます。プロファイリング版は `clone()` した複製に描いて `return img;` で返す作りなので、複製のぶん余分なコストがかかります。

### 問14

`frame` は「必ず1枚存在する画像を読むだけ」なので、**参照** が自然です。ポインタ `const Mat* frame` にすると、(a) 関数の中で「`nullptr` かもしれない」と毎回チェックするか、チェックせずに落ちる危険を抱えるかのどちらかになり、(b) 中身に触るたび `frame->cols` `(*frame).at(...)` と書き方が変わり、(c) 呼び出し側も `post_process(&img, ...)` と `&` を付けて回る必要があります。得るものが無いのに手間と事故の元が増えるだけです。一方 `runner` は、DPU ライブラリ（VART）の関数が `vart::Runner*` を引数に取る設計なので、こちらは **ポインタにせざるを得ません**。`main` では `unique_ptr<vart::Runner>` が runner を所有していて、`runner.get()` で生ポインタを取り出して渡しています。**「自分で選べるなら参照、ライブラリが要求するならポインタ」** と覚えてください。

- まとめ：「入力専用で大きい」→ `const 型&`、「中身を書き換えて結果を返したい」→ `型&`（非const参照）、「小さい値」→ 値渡しでもOK。参照ではできないこと（`nullptr`・付け替え・配列・C 流 API）が必要なときだけポインタ。

簡単なコード例で3つの渡し方を並べると、次のようになります。

In [ ]:
%%writefile ans4.cpp
#include <iostream>
#include <vector>

// 値渡し：v のコピーが作られる（大きいと遅い）。元は変わらない
void byValue(std::vector<int> v) {
    v[0] = 999;                 // コピーを書き換えるだけ。呼び出し側は変わらない
}

// const参照：コピーしない（速い）。中身は読むだけで書き換え不可
int sumByRef(const std::vector<int>& v) {
    int s = 0;
    for (int x : v) s += x;     // v[0] = 999; と書くとコンパイルエラー（const）
    return s;
}

// 非const参照：コピーせず、元のデータを直接書き換えられる
void doubleByRef(std::vector<int>& v) {
    for (int& x : v) x *= 2;    // 呼び出し側の v がそのまま変わる
}

int main() {
    std::vector<int> data = {1, 2, 3};

    byValue(data);
    std::cout << "byValue後 data[0] = " << data[0] << "\n";      // 1 のまま

    std::cout << "合計 = " << sumByRef(data) << "\n";            // 6（読むだけ）

    doubleByRef(data);
    std::cout << "doubleByRef後 data[0] = " << data[0] << "\n";  // 2（書き換わる）
    return 0;
}

In [ ]:
!g++ -std=c++17 ans4.cpp -o ans4 && ./ans4

- `byValue` はコピーを書き換えるだけなので呼び出し側の `data` は変わりません。`doubleByRef`（非const参照）はコピーせず **元の `data` を直接** 変更します。`sumByRef`（const参照）はコピーせず、かつ書き換えできません。サンプルの `const Mat& frame`（読むだけ）と `Mat& img`（直接描く）の違いが、これと同じ関係です。
- `sumByRef` の中に `v[0] = 999;` を書き足してコンパイルしてみてください。`const` のおかげで **コンパイルエラー** になります（「間違いを実行前に止めてくれる」のが `const` の価値です）。

**参照とポインタの違いを動かして見る**：同じ「合計を求める」関数を参照版とポインタ版で書き、ポインタにしかできない3つ（`nullptr`・付け替え・配列）を並べました。参照版に `nullptr` を渡す行のコメントを外すと、コンパイルエラーになることも確かめてください。

In [ ]:
%%writefile ans4e.cpp
#include <iostream>
#include <vector>

// 参照で受け取る：呼び出し側の v をそのまま指す。「無い」ことはあり得ない
int sumRef(const std::vector<int>& v) {
    int s = 0;
    for (int x : v) s += x;       // 値と同じ書き方で中身に触れる
    return s;
}

// ポインタで受け取る：アドレスを受け取る。nullptr（何も指していない）があり得る
int sumPtr(const std::vector<int>* p) {
    if (p == nullptr) return 0;   // ← 参照には無い「無い場合」の分岐
    int s = 0;
    for (int x : *p) s += x;      // 中身に触るには *p か p->
    return s;
}

int main() {
    std::vector<int> a = {1, 2, 3};
    std::vector<int> b = {10, 20, 30};

    // --- 呼び出し側の見た目 ---
    std::cout << "sumRef(a)  = " << sumRef(a)  << "\n";   // 値渡しと同じ見た目
    std::cout << "sumPtr(&a) = " << sumPtr(&a) << "\n";   // & でアドレスを渡す

    // --- ポインタだけができること① 「無い」を渡す ---
    std::cout << "sumPtr(nullptr) = " << sumPtr(nullptr) << "\n";
    // sumRef(nullptr) はコンパイルエラー。参照は必ず実体を指す

    // --- ポインタだけができること② 指す先を途中で変える ---
    const std::vector<int>* p = &a;
    std::cout << "p -> a : " << sumPtr(p) << "\n";
    p = &b;                                              // 指す先を b に付け替える
    std::cout << "p -> b : " << sumPtr(p) << "\n";

    const std::vector<int>& r = a;                       // 参照は最初に指した a から変えられない
    std::cout << "r -> a : " << sumRef(r) << "\n";
    // r = b; と書いても「r が b を指す」のではなく「a に b の中身をコピーする」意味になる（const なのでエラー）

    // --- ポインタだけができること③ 配列の先頭アドレス＋個数で扱う ---
    int buf[4] = {5, 6, 7, 8};
    int* q = buf;                                        // 配列名は先頭要素のアドレス
    std::cout << "q[0]=" << q[0] << "  *(q+2)=" << *(q + 2) << "\n";   // アドレス計算ができる
    return 0;
}

In [ ]:
!g++ -std=c++17 ans4e.cpp -o ans4e && ./ans4e

**速度の差を実際に測る**：「画像1枚」のつもりの大きな `vector`（約12MB）を100回渡します。値渡しは毎回12MBコピーするので、はっきり差が出ます（時間の測り方は次の第5章で扱います）。

In [ ]:
%%writefile ans4b.cpp
#include <iostream>
#include <vector>
#include <chrono>
using namespace std::chrono;

// 「画像1枚」のつもりの大きなデータ（int 300万個 ≒ 12MB）
long byValue(std::vector<int> v)        { return v[0]; }   // 毎回コピーされる
long byRef(const std::vector<int>& v)   { return v[0]; }   // コピーされない

int main() {
    std::vector<int> img(3000000, 1);
    const int N = 100;                   // 100 回呼ぶ
    long s = 0;

    auto t0 = system_clock::now();
    for (int i = 0; i < N; i++) s += byValue(img);
    auto t1 = system_clock::now();
    for (int i = 0; i < N; i++) s += byRef(img);
    auto t2 = system_clock::now();

    std::cout << "値渡し   " << N << "回: " << duration_cast<milliseconds>(t1 - t0).count() << " ms\n";
    std::cout << "参照渡し " << N << "回: " << duration_cast<milliseconds>(t2 - t1).count() << " ms\n";
    return s == 0;   // s を使って最適化で消されないようにしているだけ
}

In [ ]:
!g++ -std=c++17 ans4b.cpp -o ans4b && ./ans4b

**補足：スレッドに渡すときも同じ話 ―― `std::ref`**

**スレッド** とは、1つのプログラムの中で **独立して動ける処理の流れ** のことです。普通のプログラムは `main` から始まる1本の流れだけで動きますが、`std::thread` で流れを増やすと、複数の処理を **同時に（並行・並列に）** 進められます。study版は「動画の読み込み」「DPU での推論」「画面表示」をそれぞれ別のスレッドにして、同時に動かしています（スレッドを使って速くする話そのものは、続きの **cpp-lab** で扱います。ここでは「関数を別の流れで動かす仕組み」とだけ思ってください）。

study版の `main` は `thread(runYOLO, runner.get(), ref(fr), ref(shw))` のようにスレッドを作っています。`std::thread` は「第1引数の関数を、残りの引数を渡して、新しい流れで実行する」という意味です。この `ref(...)` は `std::ref` で、意味はここまでの「値渡しか参照渡しか」とまったく同じです。

- `std::thread` は、渡した引数を **既定でコピー** してからスレッドに渡します（値渡し）。関数の仮引数を `std::queue<int>& q` と参照で書いてあっても、`std::thread` の側でコピーが作られてしまいます。
- キュー `fr` / `shw` のように **複数のスレッドで1つのものを共有したい** 変数は、`std::ref(fr)` で包んで「コピーせず、本物への参照を渡せ」と指示します。
- 付け忘れると、各スレッドが **自分専用のコピー** を持ってしまい、あるスレッドが `push` したものを別のスレッドが `pop` できません。コンパイルは通ることが多いので、気づきにくいバグになります。

次の2つのセルは、`1〜6` を入れたキューを2つのスレッドで分担して処理する例です。1つ目は `std::ref` あり、2つ目は同じコードから `std::ref` を外しただけです。**実行前に、それぞれ合計何個処理され、元の `q` に何個残るか予想してください。**

（`std::mutex` / `lock_guard` は「キューを同時に触らないための鍵」です。ここでは読み飛ばして構いません。スレッドを使って速くする話、鍵の話は続きの **cpp-lab** で扱います。コンパイル時の `-pthread` はスレッドを使うプログラムに必要な指定です。）

In [ ]:
%%writefile ans4c.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <string>

// キューから1つ取り出して処理する。空になったら終了
void worker(int id, std::queue<int>& q, std::mutex& m) {
    while (true) {
        int item;
        {
            std::lock_guard<std::mutex> lock(m); // 触るのは一度に1スレッドだけ
            if (q.empty()) return;
            item = q.front();
            q.pop();
        }
        // 1行分をまとめて出す（複数スレッドが同時に表示すると行が混ざることがあるため）
        std::string line = "thread " + std::to_string(id) + " が " + std::to_string(item) + " を処理\n";
        std::cout << line;
    }
}

int main() {
    std::queue<int> q;                    // 共有するキュー（1個だけ）
    std::mutex m;
    for (int i = 1; i <= 6; i++) q.push(i);

    std::thread t1(worker, 1, std::ref(q), std::ref(m)); // std::ref で「同じ q」を渡す
    std::thread t2(worker, 2, std::ref(q), std::ref(m));
    t1.join();
    t2.join();
    std::cout << "元の q に残っている個数 = " << q.size() << "\n";   // 0（2人で6個を分担した）
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans4c.cpp -o ans4c && ./ans4c

In [ ]:
%%writefile ans4d.cpp
#include <iostream>
#include <thread>
#include <queue>
#include <mutex>
#include <string>

// ans4c.cpp と同じ worker。ただし q を「コピー」で受け取る（& を外した）
void worker(int id, std::queue<int> q, std::mutex& m) {
    while (true) {
        int item;
        {
            std::lock_guard<std::mutex> lock(m);
            if (q.empty()) return;
            item = q.front();
            q.pop();
        }
        std::string line = "thread " + std::to_string(id) + " が " + std::to_string(item) + " を処理\n";
        std::cout << line;
    }
}

int main() {
    std::queue<int> q;
    std::mutex m;
    for (int i = 1; i <= 6; i++) q.push(i);

    std::thread t1(worker, 1, q, std::ref(m));   // std::ref なし → q のコピーが各スレッドに渡る
    std::thread t2(worker, 2, q, std::ref(m));
    t1.join();
    t2.join();
    std::cout << "元の q に残っている個数 = " << q.size() << "\n";   // 6 のまま（誰も本物を触っていない）
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ans4d.cpp -o ans4d && ./ans4d

- `std::ref` あり：2つのスレッドが **同じ `q`** から取り出すので、合計ちょうど6個が処理され、最後に `q` は空（0個）になります。どちらのスレッドが何個取るかは実行のたびに変わります。
- `std::ref` なし：各スレッドが **6個入りのコピー** を受け取るので、`thread 1` も `thread 2` も6個ずつ、合計12個処理します。元の `q` には6個が **そのまま残ります**。分担になっていません。
- まとめると、**関数の引数の `&` と、`std::thread` に渡すときの `std::ref` は、両方そろって初めて「本物を共有」できます。** study版の `ref(fr)`, `ref(shw)` はこのためのものです。

# 第5章　`std::chrono` で処理時間を測る（プロファイリングの核心）

## 解説

プロファイリング版は、各処理の **前後で時刻を取得し、その差** を `[mS]`（ミリ秒）で出力します。例：`runYOLO preprocessing time= ... [mS]`、`runYOLO dpu time= ...`、`runYOLO post time= ...`。これが「どの処理が重いか」を測る仕組みそのものです。

- `#include <chrono>`。`system_clock::now()` で **今の時刻** を取る。
- `t1 - t0` は「時間の長さ（duration）」。`duration_cast<milliseconds>(...)` でミリ秒に変換し、`.count()` で数値を取り出す。

---

## 設問

### 問15

`system_clock::now()` は何を返している？

### 問16

`duration_cast<milliseconds>(t1 - t0).count()` は全体として何を計算している？

### 問17

プロファイリング版が、まとめて1回ではなく **前処理・DPU・後処理・表示を別々に** 測っているのはなぜ？

ループ回数（`100000000L`）を10倍・1/10倍に変えて、表示される時間がどう変わるかも見てみましょう。

**同じセルを何度か実行してみてください。** 同じプログラムなのに、表示される時間は毎回少しずつ違います（Colab では数十 ms 程度ばらつきます）。Colab の CPU は他の利用者のプログラムと共有されていて、OS が他の処理に切り替えたり、CPU の動作クロックが変わったりするためです。本番の KV260 でも同じことが起きるので、**時間は1回ではなく何回か測って傾向を見る** のが基本です。

In [ ]:
%%writefile ex5.cpp
#include <iostream>
#include <chrono>
using namespace std::chrono;

int main() {
    auto t0 = system_clock::now();
    // ここに時間を測りたい処理を書く（例：大きめのループ）
    long sum = 0;
    for (long i = 0; i < 100000000L; i++) sum += i;
    auto t1 = system_clock::now();

    std::cout << duration_cast<milliseconds>(t1 - t0).count() << " ms\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ex5.cpp -o ex5 && ./ex5

## 設問の解答

### 問15

`system_clock::now()` は **その瞬間の時刻（時計の現在値）** を返します。

### 問16

`t1 - t0` は2つの時刻の差＝**経過時間（duration）**。それを `duration_cast<milliseconds>(...)` でミリ秒単位に変換し、`.count()` で **数値（整数）として取り出して** います。全体で「処理にかかったミリ秒」を求めています。

### 問17

まとめて測ると「全体で何msか」しか分かりませんが、**処理ごとに分けて測る** と「どの処理が一番重いか（ボトルネックはどこか）」が分かります。これがプロファイリングの目的で、本番でボトルネックを見つける判断材料になります。

- 補足：`system_clock` も `steady_clock` も `<chrono>` ヘッダの中にある **時計の種類** です（`chrono` はライブラリの名前、`system_clock` はその中の時計の1つ）。
  - `system_clock` … **壁掛け時計**。OS の現在時刻そのもので、`time_t` に変換して日付・時刻として表示できる。ただし NTP による時刻合わせや手動の時刻変更で **進んだり戻ったりする** ことがある。
  - `steady_clock` … **ストップウォッチ**。起動からの経過時間のような単調増加の時計で、絶対に戻らない。日付には変換できないが、**経過時間の計測にはこちらが正しい選択**。
  - サンプルは `system_clock` を使っています。KV260 が起動直後に NTP で時刻合わせをする瞬間などに当たると、測った差がおかしな値になる可能性があります。自分で書くときは `steady_clock::now()` に置き換えるだけで済むので、計測には `steady_clock` を使うことをすすめます（cpp-lab の演習はすべて `steady_clock` です）。

# 第6章　関数とグローバル変数・スコープ

## 解説

サンプルは `readFrame` / `displayFrame` / `runYOLO` / `post_process` のように処理を関数に分けています。また `shapes` や `start_time` のように、複数の関数から使う値は **グローバル変数** として宣言されています。

- 関数は `戻り値の型 関数名(引数) { ... return ...; }`。
- グローバル変数は **関数の外** で宣言する。どの関数からも読み書きできる。
- ローカル変数は関数（ブロック `{ }`）の中で宣言し、その中でだけ有効。

---

## 設問

### 問18

2つの `int` を受け取って和を返す関数 `int add(int a, int b)` を定義し、`main` から呼んで結果を表示するには？

### 問19

「グローバル変数」と「関数の中だけで使えるローカル変数」の違いを一言で説明すると？

### 問20

第5章の `system_clock` を使って、`add` を `for` 文で何回も呼び、**全部終わるまでの所要時間** を測るには？（「`add` を呼んだ回数」は **グローバル変数** に記録してみましょう）

**ヒント**：`int` を1回足すのはとても速いので、`for` の回数はある程度多く（例：1000万〜1億回）しないとミリ秒では測れません。

In [ ]:
%%writefile ex6.cpp
#include <iostream>
#include <chrono>
using namespace std::chrono;

// 3. add を呼んだ回数を記録するグローバル変数をここに宣言する

// 1. 2つの int を受け取って和を返す関数 add を定義する

int main() {
    // 1. add(3, 4) を呼んで結果を表示する

    // 3. system_clock で時刻を取り、for 文で add を何回も呼び、所要時間と呼んだ回数を表示する

    return 0;
}

In [ ]:
!g++ -std=c++17 ex6.cpp -o ex6 && ./ex6

## 設問の解答

### 問18

**`add` の定義と呼び出し（基本形）**

In [ ]:
%%writefile ans6a.cpp
#include <iostream>

int add(int a, int b) {              // 関数定義（値渡し）
    return a + b;
}

int main() {
    std::cout << add(3, 4) << "\n";  // 7
    return 0;
}

In [ ]:
!g++ -std=c++17 ans6a.cpp -o ans6a && ./ans6a

### 問19

**グローバル変数とローカル変数の違い**

**グローバル変数** はプログラム全体（どの関数からも）アクセスできる変数、**ローカル変数** はその関数（ブロック）の中だけで有効な変数です。サンプルが `shapes` をグローバルにしているのは、複数の関数から同じ形状情報を参照するためです（ただしグローバル変数は多用すると管理が難しくなるので、必要な範囲にとどめるのが基本です）。

### 問20

**グローバル変数 ＋ `system_clock` で所要時間も測る**

In [ ]:
%%writefile ans6b.cpp
#include <iostream>
#include <chrono>
using namespace std::chrono;

long g_callCount = 0;                // グローバル変数：add が呼ばれた回数を記録

int add(int a, int b) {
    g_callCount++;                   // どの関数からでも触れる（グローバル）
    return a + b;
}

int main() {
    const int N = 100000000;         // ローカル変数（main の中だけ有効）
    long sum = 0;                    // ローカル変数

    auto t0 = system_clock::now();
    for (int i = 0; i < N; i++)
        sum = add(sum % 1000, i % 1000);   // add を N 回呼ぶ
    auto t1 = system_clock::now();

    std::cout << "sum = " << sum << "\n";
    std::cout << "add を呼んだ回数(global) = " << g_callCount << "\n";
    std::cout << "所要時間 = "
              << duration_cast<milliseconds>(t1 - t0).count() << " ms\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ans6b.cpp -o ans6b && ./ans6b

- ここでは `g_callCount`（グローバル）で「何回呼んだか」を全体で数え、`N` や `sum`（ローカル）は `main` の中だけで使っています。グローバルとローカルの両方が登場する例です。
- 注意：`int` の足し算1回はとても速いので、`for` を100〜1000回程度にすると **所要時間はほぼ 0 ms** と表示されます。意味のある時間を出すには、上のように **1000万〜1億回** といった大きな回数にするか、`milliseconds` の代わりに `microseconds` / `nanoseconds` で測ります。

# 第7章　クラス・テンプレート・ファンクタ

## 解説

サンプルの `class paircomp { bool operator()(...) ... };` は「**関数のように呼べるオブジェクト**（ファンクタ）」で、`priority_queue` の並び替え基準として使われています。`template<typename T> class concurrent_queue` は「**中身の型を後から決められる**」テンプレートクラスです。

- ファンクタ：クラスに `operator()` を定義すると、オブジェクトを `m(3, 4)` のように **関数として呼べる**。
- テンプレート：クラス定義の直前に `template<typename T>` を書き、型の代わりに `T` を使う。`Box<int>`、`Box<std::string>` のように使うときに型を決める。

---

## 設問

### 問21

`operator()` を持つクラス `Mul` を作り、`Mul m; std::cout << m(3, 4);` で `12` と表示できるようにするには？（`m(3,4)` が関数呼び出しのように動く）

### 問22

`template<typename T>` を使って、任意の型の値を1つ保持して取り出せる小さなクラス `Box<T>` を作るには？（例：`Box<int> b(5); b.get();` で `5`）

**ヒント**：ファンクタは `class Mul { public: int operator()(int a, int b){ return a*b; } };`。

In [ ]:
%%writefile ex7.cpp
#include <iostream>
#include <string>

// 1. operator() を持つクラス Mul を作る（m(3, 4) で 12）

// 2. template<typename T> を使って、値を1つ保持して get() で取り出せる Box<T> を作る

int main() {
    // Mul m; std::cout << m(3, 4) << "\n";
    // Box<int> b(5); std::cout << b.get() << "\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ex7.cpp -o ex7 && ./ex7

## 設問の解答

### 問21

`class Mul { public: int operator()(int a, int b) { return a * b; } };` のように `operator()` を定義すると、`Mul m; m(3, 4)` が関数呼び出しのように動いて `12` を返します。

### 問22

クラス定義の直前に `template<typename T>` を書き、値を `T value_;` で持って `T get()` で返します。`Box<int> b(5); b.get();` で `5` になります。

問21・問22 をまとめたプログラムが次のセルです。

In [ ]:
%%writefile ans7a.cpp
#include <iostream>
#include <string>

// 1. ファンクタ（() で呼べるクラス）
class Mul {
public:
    int operator()(int a, int b) { return a * b; }
};

// 2. テンプレートクラス
template<typename T>
class Box {
    T value_;
public:
    Box(T v) : value_(v) {}
    T get() { return value_; }
};

int main() {
    Mul m;
    std::cout << m(3, 4) << "\n";      // 12  ← m(...) が呼び出せる

    Box<int> bi(5);
    Box<std::string> bs("hi");
    std::cout << bi.get() << " " << bs.get() << "\n"; // 5 hi
    return 0;
}

In [ ]:
!g++ -std=c++17 ans7a.cpp -o ans7a && ./ans7a

- `operator()` を定義すると、オブジェクトをあたかも関数のように `m(3,4)` と呼べます。
- `template<typename T>` を付けると、`Box<int>` でも `Box<std::string>` でも同じコードが使えます。サンプルの `concurrent_queue<imagePair>` も、中身の型を `imagePair` に決めて使っているテンプレートクラスです。

**補足：ファンクタが `priority_queue` の順番を決める仕組み（サンプルの `paircomp`）**

`priority_queue`（優先度つきキュー）は、要素を取り出すとき **比較ファンクタの `operator()` を呼んで「どちらを先にするか」を決めます**。サンプルの `paircomp` は、フレーム番号（`pair` の `.first`）が小さいものを先に取り出すための比較です。

**宣言の読み方**：`priority_queue` のテンプレート引数は3つあります。

```cpp
std::priority_queue< 要素の型,  内部で要素を保管する入れ物の型,  比較のしかた > pq;
```

- 1つ目 … キューに入れる要素の型（ここでは `std::pair<int,int>`）
- 2つ目 … `priority_queue` が内部で要素を並べておくのに使う入れ物（コンテナ）。ほぼ常に `std::vector<要素の型>`
- 3つ目 … 「どちらを先に取り出すか」を決めるファンクタ（ここでは `paircomp`）

普段 `std::priority_queue<int> pq;` と短く書けるのは、2つ目と3つ目に **省略時の値**（`std::vector<int>` と `std::less<int>`＝`<` で比べて大きい順）があるからです。テンプレート引数は関数の既定引数と同じで **後ろからしか省略できない** ため、3つ目の比較だけを変えたいときも、2つ目の入れ物を自分で書く必要があります。それでこの長い宣言になります。長くて読みにくいので、次の例では第3章の `using` で要素の型に `Item` という別名を付けています。

In [ ]:
%%writefile ans7b.cpp
#include <iostream>
#include <queue>
#include <vector>
#include <utility>

using Item = std::pair<int,int>;    // 要素の型に別名を付ける（問3）。以下が読みやすくなる

// .first が小さい要素を先に取り出すための比較ファンクタ
class paircomp {
public:
    bool operator()(const Item& a, const Item& b) const {
        return a.first > b.first;   // 「a を b より後ろにする」条件
    }
};

int main() {
    // priority_queue のテンプレート引数は3つ：
    //   <要素の型, 内部で要素を保管する入れ物の型, 比較のしかた>
    // 3つ目（比較）を変えたいので、途中の2つ目（入れ物）も省略せずに書く
    std::priority_queue<Item,               // 要素の型
                        std::vector<Item>,  // 入れ物（ほぼ常に vector<要素の型>）
                        paircomp> pq;       // 比較ファンクタ
    // {3, 300} は std::pair<int,int>{3, 300} の短い書き方（波カッコで pair を作る）
    pq.push({3, 300});          // std::make_pair(3, 300) と書いても同じ
    pq.push({1, 100});
    pq.push({2, 200});

    while (!pq.empty()) {
        auto p = pq.top();          // first が一番小さいものが出てくる
        pq.pop();
        std::cout << p.first << " => " << p.second << "\n";
    }
    // 出力: 1 => 100 / 2 => 200 / 3 => 300（first の昇順）
    return 0;
}

In [ ]:
!g++ -std=c++17 ans7b.cpp -o ans7b && ./ans7b

- **`pq.push({3, 300})` の `{ }` について**：`pq` は `std::pair<int,int>` を入れるキューなので、`push` には「pairを1個」渡す必要があります。`{3, 300}` は **「`first` が3、`second` が300の `std::pair<int,int>` を作る」** という波カッコ（中カッコ）の書き方で、`std::pair<int,int>{3, 300}` や `std::make_pair(3, 300)` と同じ意味です。`push(std::make_pair(3, 300))` と書いても全く同じ動きになります（`{ }` の方が短いので使っているだけ）。`{3, 300}` のように **2つの値を波カッコでまとめると pair（2つ組）になる** と覚えてください。
- `priority_queue` の3つ目の型引数に `paircomp` を指定すると、要素を並べるたびに `paircomp` の `operator()(a, b)` が呼ばれます。`return a.first > b.first;` は「`a.first` が大きいなら `a` を後回しにする」という意味なので、結果として **`first` が小さい順** に取り出せます。
- サンプルでは、並列に処理して **バラバラの順序で完成したフレーム** を、この仕組みで **フレーム番号順に並べ直して** から表示しています。`operator()` の中身（`>` か `<` か、何を比べるか）を変えれば、取り出す順番を自由に変えられます。試しに `>` を `<` に変えて実行してみてください（降順になります）。

# 第8章　動的メモリと `unique_ptr`

## 解説

サンプルは出力バッファを `int8_t* result0 = new int8_t[size0];` のように **動的確保** し、最後に `delete[] result0;` で解放します。また DPU入出力のバッファは `std::make_unique<CpuFlatTensorBuffer>(...)`（`unique_ptr`）で管理しています。

**なぜここは `std::vector` ではなく `new` なのか。** DPU を動かすライブラリ（VART）の API が、バッファを「先頭アドレス（生のポインタ）＋サイズ」の形で受け取る C 流の設計になっていて、サンプルはそれに合わせて `new int8_t[size]` で確保した生ポインタをそのまま渡しているためです。ただし、これは **そう書かなければならない** という意味ではありません。`std::vector<int8_t> buf(size);` で確保して `buf.data()` を渡せば同じ先頭アドレスが得られ、解放は自動で、第2章の補足で見た「`new[]` は初期化されない」問題も起きません。つまりサンプルの `new` は歴史的な書き方（元の Xilinx サンプルが C 流）であって、自分で書き足す部分では `vector` を使って構いません。この章は、そういう **既存コードの `new`/`delete` を読んで、正しく扱える** ようになるためのものです。

- `new`↔`delete`、`new[]`↔`delete[]` が対。確保したら **必ず** 対になる方で解放する。
- スマートポインタは `#include <memory>`。`std::make_unique<型>(...)` で作ると、スコープを抜けるとき **自動で解放** されます。

---

## 設問

### 問23

`new[]`（配列の動的確保）で作った領域は、`delete` と `delete[]` のどちらで解放するのが正しい？ 解放を忘れると何が起きる？

### 問24

`std::unique_ptr` を使うと、生のポインタ（`new`/`delete`）に比べて何が嬉しい？

### 問25

（観察）プロファイリング版には `new int8_t[...]` に対して `delete imageInputs;`（`[]` なし）と書かれた箇所があります。`new[]` で確保した配列の **正しい** 解放はどう書くべき？

この章は考える問題です。答えを書いてから、解答のコードを動かして確かめてください。

## 設問の解答

### 問23

`new[]` で確保した配列は **`delete[]`** で解放します。解放を忘れると **メモリリーク**（使われないメモリが増え続ける）になり、長時間動かす動画処理では特に問題になります。

### 問24

`unique_ptr` は **スコープを抜けると自動的に解放** してくれるので、`delete` の書き忘れや、途中で `return`／例外が起きたときの解放漏れを防げます（「誰が解放するか」が明確になる）。

### 問25

`new int8_t[...]`（配列）に対する正しい解放は **`delete[] imageInputs;`** です。`delete imageInputs;`（`[]` なし）は配列に対しては不正で、本来は `delete[]` と書くべきものです。

- 原則：`new`↔`delete`、`new[]`↔`delete[]` を必ず対で使う。可能なら最初から `std::vector` や `unique_ptr` を使うと、こうした解放ミス自体が起きにくくなります。

次のセルは、確保・解放のタイミングが見えるようにした例です。`unique_ptr` の方は `delete` を **一行も書いていない** のに「解放」が表示されることを確認してください。

In [ ]:
%%writefile ans8.cpp
#include <iostream>
#include <memory>

struct Buf {                         // 確保・解放が見えるように表示するだけの型
    Buf()  { std::cout << "  確保\n"; }
    ~Buf() { std::cout << "  解放\n"; }
};

void rawPointer() {
    std::cout << "[生のポインタ]\n";
    Buf* p = new Buf;
    // ... ここで return や例外があると delete に届かず「解放」が出ない（メモリリーク）
    delete p;                        // 自分で書かないと解放されない
}

void smartPointer() {
    std::cout << "[unique_ptr]\n";
    auto p = std::make_unique<Buf>();
    // delete を書かない。関数を抜けるとき（スコープの終わり）に自動で解放される
}

int main() {
    rawPointer();
    smartPointer();

    // 配列は new[] ↔ delete[] が対
    int8_t* arr = new int8_t[1024];
    arr[0] = 1;
    delete[] arr;                    // delete arr; は誤り

    std::cout << "done\n";
    return 0;
}

In [ ]:
!g++ -std=c++17 ans8.cpp -o ans8 && ./ans8

- 試しに `rawPointer()` の `delete p;` をコメントアウトして実行すると、生のポインタ側だけ「解放」が出なくなります。これがメモリリークです（プログラムは何事もなく終わるので **気づきにくい** のが厄介な点です）。